# 1. Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 2. Comando Linux (NAO EXECUTAR)

In [ ]:
!rm *.csv

In [ ]:
!wget https://dados.transportes.gov.br/dataset/42e2320b-ea67-4fdc-896f-71363e043fc6/resource/da6babad-68a6-461b-89f9-e06fd19ef7e6/download/renaest_dabertos_20260112.zip

In [ ]:
!unzip renaest_dabertos_20260112.zip

In [ ]:
!rm *.zip

# 3. Atribuindo os dataframes (NAO EXECUTAR)

## 3.1 Criando os dataframes_raw

In [ ]:
# Criando os dataframes
df_acidentes_raw = pd.read_csv("Acidentes_DadosAbertos_20260112.csv", sep=";")
df_tipo_veiculo_raw = pd.read_csv("TipoVeiculo_DadosAbertos_20260112.csv", sep=";")
df_localidades_raw = pd.read_csv("Localidade_DadosAbertos_20260112.csv", sep=";")
df_viitmas_raw = pd.read_csv("Vitimas_DadosAbertos_20260112.csv", sep=";")

In [ ]:
# Analisando o dataframe
df_acidentes_raw.head()

In [ ]:
# Dicionário com as capitais de cada UF
capitais = {
    'AC': 'Rio Branco', 'AL': 'Maceió', 'AP': 'Macapá', 'AM': 'Manaus', 'BA': 'Salvador', 'CE': 'Fortaleza',
    'DF': 'Brasília', 'ES': 'Vitória', 'GO': 'Goiânia', 'MA': 'São Luís', 'MT': 'Cuiabá', 'MS': 'Campo Grande',
    'MG': 'Belo Horizonte', 'PA': 'Belém', 'PB': 'João Pessoa', 'PR': 'Curitiba', 'PE': 'Recife', 'PI': 'Teresina',
    'RJ': 'Rio de Janeiro', 'RN': 'Natal', 'RS': 'Porto Alegre', 'RO': 'Porto Velho', 'RR': 'Boa Vista',
    'SC': 'Florianópolis', 'SP': 'São Paulo', 'SE': 'Aracaju', 'TO': 'Palmas'
}

import unicodedata

# Função para normalizar texto (remover acentos e lowercase)
def normalizar(texto):
    if pd.isna(texto): return ""
    return ''.join(c for c in unicodedata.normalize('NFD', str(texto)) if unicodedata.category(c) != 'Mn').lower().strip()

# Preparar df de localidades para obter códigos IBGE
# Selecionamos apenas as colunas necessárias e removemos duplicatas de municípios
df_loc_unique = df_localidades_raw[['codigo_ibge', 'municipio', 'uf']].drop_duplicates()
df_loc_unique['municipio_norm'] = df_loc_unique['municipio'].apply(normalizar)

# Identificar códigos IBGE das capitais
codigos_capitais = []
for uf, capital in capitais.items():
    capital_norm = normalizar(capital)
    # Busca o código IBGE correspondente à UF e ao nome da capital normalizado
    cods = df_loc_unique[
        (df_loc_unique['uf'] == uf) &
        (df_loc_unique['municipio_norm'] == capital_norm)
    ]['codigo_ibge'].unique()
    codigos_capitais.extend(cods)

# Filtrar o dataframe de acidentes
df_acidentes_capitais_raw = df_acidentes_raw[df_acidentes_raw['codigo_ibge'].isin(codigos_capitais)].copy()

# Adicionar o nome do município ao dataframe de acidentes (merge)
# Removemos a coluna 'municipio_norm' auxiliar antes do merge
df_loc_merge = df_loc_unique[['codigo_ibge', 'municipio', 'uf']].drop_duplicates()
df_acidentes_capitais_raw = df_acidentes_capitais_raw.merge(df_loc_merge, on='codigo_ibge', how='left')

display(df_acidentes_capitais_raw.head())
print(f"Total de acidentes em capitais: {len(df_acidentes_capitais_raw)}")

In [ ]:
# Remover anos incompletos (2018 e 2025) do dataframe
df_acidentes_capitais_raw = df_acidentes_capitais_raw[~df_acidentes_capitais_raw['ano_acidente'].isin([2018, 2025])].copy()

# Verificar os anos restantes e o tamanho do dataframe
print("Anos restantes:", sorted(df_acidentes_capitais_raw['ano_acidente'].unique()))
print(f"Total de acidentes após remoção: {len(df_acidentes_capitais_raw)}")

In [ ]:
# Calcular a porcentagem de valores nulos por coluna
null_percentages = (df_acidentes_capitais_raw.isnull().mean()) * 100

# Criar uma lista de tuplas (coluna, porcentagem)
null_info_tuples = list(zip(null_percentages.index, null_percentages.values))

# Exibir a lista de tuplas
display(null_info_tuples)

# Exibir ordenado para facilitar a leitura
null_info_sorted = sorted(null_info_tuples, key=lambda x: x[1], reverse=True)
display(pd.DataFrame(null_info_sorted, columns=['Coluna', '% Nulos']))

In [ ]:
# Colunas com valores nulos identificadas anteriormente
cols_com_nulos = ['latitude_acidente', 'longitude_acidente', 'bairro_acidente', 'end_acidente']

# Calcular a porcentagem de nulos por município e UF
nulos_por_municipio = df_acidentes_capitais_raw.groupby(['uf', 'municipio'])[cols_com_nulos].apply(lambda x: x.isnull().mean() * 100)

# Ordenar pela latitude para ver quem tem menos dados de geolocalização
nulos_por_municipio_sorted = nulos_por_municipio.sort_values(by='latitude_acidente', ascending=False)

# Exibir a tabela
display(nulos_por_municipio_sorted)

# Visualizar com um heatmap
plt.figure(figsize=(10, 12))
sns.heatmap(nulos_por_municipio_sorted, annot=True, cmap='viridis_r', fmt='.1f')
plt.title('Porcentagem de Valores Nulos por Capital')
plt.ylabel('UF - Município')
plt.show()

In [ ]:
# Mapeamento de UFs para Regiões
regioes_map = {
    'AC': 'Norte', 'AP': 'Norte', 'AM': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
}

# Criar coluna de região no dataframe raw baseada na UF do acidente
df_acidentes_capitais_raw['regiao'] = df_acidentes_capitais_raw['uf_acidente'].map(regioes_map)

# Colunas com valores nulos identificadas anteriormente
cols_com_nulos = ['latitude_acidente', 'longitude_acidente', 'bairro_acidente', 'end_acidente']

# Calcular a porcentagem de nulos por região
nulos_por_regiao = df_acidentes_capitais_raw.groupby('regiao')[cols_com_nulos].apply(lambda x: x.isnull().mean() * 100)

# Exibir a tabela
display(nulos_por_regiao)

# Visualizar com um heatmap para facilitar a identificação
plt.figure(figsize=(10, 6))
sns.heatmap(nulos_por_regiao, annot=True, cmap='viridis_r', fmt='.1f')
plt.title('Porcentagem de Valores Nulos por Região nas Capitais')
plt.ylabel('Região')
plt.show()

In [ ]:
# Preparar dados: Agrupar por município, UF e tipo de acidente
# Pivotar para que os tipos de acidentes virem colunas (necessário para barras empilhadas)
acidentes_por_cidade_tipo = df_acidentes_capitais_raw.groupby(['municipio', 'uf', 'tp_acidente']).size().unstack(fill_value=0)

# Calcular o total para ordenação
acidentes_por_cidade_tipo['Total'] = acidentes_por_cidade_tipo.sum(axis=1)

# Ordenar do menor para o maior total
acidentes_sorted = acidentes_por_cidade_tipo.sort_values(by='Total', ascending=True)

# Separar os totais para uso nos rótulos e remover da plotagem
totals = acidentes_sorted['Total']
df_plot = acidentes_sorted.drop(columns='Total')

# Ajustar o índice para exibir "Cidade - UF"
df_plot.index = [f"{m} - {u}" for m, u in df_plot.index]

# Exibir os totais
display(totals)

# Criar gráfico de barras horizontal empilhado
ax = df_plot.plot(kind='barh', stacked=True, figsize=(14, 12), colormap='tab20')

# Adicionar rótulos de dados (Total) ao final das barras
for i, v in enumerate(totals):
    ax.text(v, i, f' {v:,.0f}'.replace(',', '.'), va='center', fontsize=9)

plt.title('Quantidade de Acidentes por Capital e Tipo')
plt.xlabel('Quantidade de Acidentes')
plt.ylabel('Capital')

# Posicionar a legenda fora do gráfico para não poluir
plt.legend(title='Tipo de Acidente', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Calcular a distribuição dos tipos de acidentes (Top 15)
tipos_acidentes = df_acidentes_capitais_raw['tp_acidente'].value_counts().head(15)

# Calcular porcentagens
total = len(df_acidentes_capitais_raw)
percentuais = (tipos_acidentes / total) * 100

# Configurar o gráfico
plt.figure(figsize=(12, 8))
# Correção: atribuir y a hue e legend=False para evitar FutureWarning
ax = sns.barplot(x=tipos_acidentes.values, y=tipos_acidentes.index, hue=tipos_acidentes.index, palette='viridis', legend=False)

plt.title('Distribuição dos Tipos de Acidentes nas Capitais (Top 15)')
plt.xlabel('Quantidade de Acidentes')
plt.ylabel('Tipo de Acidente')

# Adicionar rótulos de dados (Quantidade e Porcentagem)
for i, v in enumerate(tipos_acidentes.values):
    perc = percentuais.iloc[i]
    ax.text(v + 100, i, f'{v:,.0f} ({perc:.1f}%)'.replace(',', '.'), va='center', fontsize=10)

plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 3.2 Analise df_acidentes_capitais_raw

In [ ]:
# Analise df_acidentes_capitais_raw

display(df_acidentes_capitais_raw.head())
display(df_acidentes_capitais_raw.info())
display(df_acidentes_capitais_raw['chv_localidade'].unique())

In [ ]:
# Criando a lista e chv_localidade
lista_chv_localidade = df_acidentes_capitais_raw['chv_localidade'].unique()

## 3.3 Analisando df_vitimas e criando a df_vitimas_capitais


In [ ]:
# Analise df_viitmas_raw

display(df_viitmas_raw.head())
display(df_viitmas_raw['tp_envolvido'].unique())
display(df_viitmas_raw['genero'].unique())
display(df_viitmas_raw['faixa_idade'].unique())
display(df_viitmas_raw['susp_alcool'].unique())

In [ ]:
df_vitimas_capitais = df_viitmas_raw.query("chv_localidade in @lista_chv_localidade")

In [ ]:
df_vitimas_capitais.to_csv("df_vitimas_capitais.csv", sep=";")

## 3.4 Analisando df_tipo_veiculos_raw e criando a df_tipo_veiculo_capitais

In [ ]:
# Analise df_tipo_veiculos_raw

display(df_tipo_veiculo_raw[df_tipo_veiculo_raw["num_acidente"] == 4960250])
display(df_tipo_veiculo_raw.head())
display(df_tipo_veiculo_raw['tipo_veiculo'].unique())
display(df_tipo_veiculo_raw.info())
display(df_tipo_veiculo_raw.count())

In [ ]:
# Criando a lista de num_acidente do dataframe df_acidentes_capitais_raw

lista_num_acidente = df_acidentes_capitais_raw['num_acidente'].unique()

In [ ]:
# Criacao do dataframe de tipo de veiculo apenas com acidentes das capitais.

df_tipo_veiculo_capitais = df_tipo_veiculo_raw[df_tipo_veiculo_raw["num_acidente"].isin(lista_num_acidente)]

# 4. Analisando as bases de dados das capitais

## 4.1. Atribuindo df_capitais_raw a um dataframe

In [ ]:
# Atribuindo df_capitais_raw.csv ao dataframe df_acidentes_capitais_raw

df_acidentes_capitais_raw = pd.read_csv("df_capitais_raw.csv", sep=";")

## 4.2. Analisando tipos de veiculos por capitais

In [ ]:
# Atribuindo o df_tipo_veiculo_capitais.csv ao dataframe

df_tipo_veiculo_capitais = pd.read_csv("df_tipo_veiculo_capitais.csv", sep=";")

In [ ]:
# Valores a serem substituídos por 'ND'
valores_para_nd_tipo_veiculo = ['DESCONHECIDO', 'NAO INFORMADO']

# Substituir os valores na coluna 'tipo_veiculo' do df_tipo_veiculo_capitais
df_tipo_veiculo_capitais['tipo_veiculo'] = df_tipo_veiculo_capitais['tipo_veiculo'].replace(valores_para_nd_tipo_veiculo, 'ND')

print("Valores 'DESCONHECIDO' e 'NAO INFORMADO' na coluna 'tipo_veiculo' do df_tipo_veiculo_capitais foram substituídos por 'ND'.")
display(df_tipo_veiculo_capitais['tipo_veiculo'].unique())

In [ ]:
# Analisando df_tipo_veiculo_capitais

display(df_tipo_veiculo_capitais.head())
display(df_tipo_veiculo_capitais.info())
display(df_tipo_veiculo_capitais.count())

In [ ]:
# Fazer um left join na df_tipo_veiculos_capitais para que pegue o UF da tabela df_acidentes_capitais_raw

df_tipo_veiculo_capitais = df_tipo_veiculo_capitais.merge(df_acidentes_capitais_raw, how="left", on="num_acidente")

In [ ]:
df_tipo_veiculo_capitais = df_tipo_veiculo_capitais[["num_acidente","tipo_veiculo", "qtde_veiculos", "municipio","uf", "regiao"]]

In [ ]:
df_tipo_veiculo_capitais

In [ ]:
df_tipo_veiculo_capitais["tipo_veiculo"].unique()

In [ ]:
# 1. Calcular o total de veículos no dataset (Total Geral)
total_veiculos = df_tipo_veiculo_capitais['qtde_veiculos'].sum()

# 2. Calcular a quantidade dos tipos 'ND' (resultante da normalização)
tipos_interesse_nd = ['ND']
counts_nd = df_tipo_veiculo_capitais[df_tipo_veiculo_capitais['tipo_veiculo'].isin(tipos_interesse_nd)].groupby('tipo_veiculo')['qtde_veiculos'].sum()

# 3. Obter valores para 'ND'
qtd_nd = counts_nd.get('ND', 0)

# 4. Calcular porcentagens
pct_nd = (qtd_nd / total_veiculos) * 100
pct_informado = ((total_veiculos - qtd_nd) / total_veiculos) * 100

# 5. Exibir resultados numéricos
print(f"Total de Veículos no Dataset: {total_veiculos:,.0f}".replace(',', '.'))
print(f"ND (Não Definido): {qtd_nd:,.0f} ({pct_nd:.2f}%)".replace(',', '.'))
print(f"Total Informado: {total_veiculos - qtd_nd:,.0f} ({pct_informado:.2f}%)".replace(',', '.'))

# 6. Visualização Gráfica (Bar Chart)
labels = ['Informado', 'ND (Não Definido)']
values = [total_veiculos - qtd_nd, qtd_nd]
percentages = [pct_informado, pct_nd]
colors = ['#99ff99', '#ff9999'] # Verde para OK, Vermelho para ND

plt.figure(figsize=(10, 6))
bars = plt.bar(labels, values, color=colors, edgecolor='grey')

# Adicionar rótulos nas barras
for bar, pct, val in zip(bars, percentages, values):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height,
             f'{val:,.0f}\n({pct:.1f}%)'.replace(',', '.'),
             ha='center', va='bottom', fontsize=12)

plt.title('Representatividade de Dados Faltantes de Veículos (Total do Dataset)', fontsize=14)
plt.ylabel('Quantidade de Veículos')
plt.ylim(0, max(values) * 1.15) # Aumentar limite Y para caber os rótulos
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# 1. Calcular o total de veículos envolvidos por cidade
total_veiculos_cidade = df_tipo_veiculo_capitais.groupby(['municipio', 'uf'])['qtde_veiculos'].sum()

# 2. Calcular a quantidade dos tipos 'ND'
tipos_interesse = ['ND'] # Agora apenas 'ND' é o tipo de interesse
df_unknown = df_tipo_veiculo_capitais[df_tipo_veiculo_capitais['tipo_veiculo'].isin(tipos_interesse)]
counts_nd = df_unknown.groupby(['municipio', 'uf', 'tipo_veiculo'])['qtde_veiculos'].sum().unstack(fill_value=0)

# Garantir que a coluna 'ND' exista
if 'ND' not in counts_nd.columns:
    counts_nd['ND'] = 0

# 3. Juntar os dados e calcular as porcentagens
df_calc = counts_nd.merge(total_veiculos_cidade.rename('Total_Geral'), left_index=True, right_index=True)

df_calc['% ND'] = (df_calc['ND'] / df_calc['Total_Geral']) * 100

# 4. Ordenar e preparar para plotagem
df_sorted = df_calc.sort_values('% ND', ascending=True)

# Selecionar apenas as colunas de porcentagem para o gráfico
df_plot = df_sorted[['% ND']]

# Ajustar o índice para exibir "Cidade - UF"
df_plot.index = [f"{m} - {u}" for m, u in df_plot.index]

# Exibir os dados tabulados (Totais e Percentuais - Top 5 Piores)
print("Tabela de Dados por Capital (Top 5 Piores):")
display(df_sorted[['Total_Geral', 'ND', '% ND']].tail())

# 5. Plotar Gráfico
ax = df_plot.plot(kind='barh', stacked=True, figsize=(14, 12), colormap='Set2')

# Adicionar rótulos de porcentagem e total na ponta da barra
for i, v in enumerate(df_sorted['% ND']):
    total_abs = df_sorted['Total_Geral'].iloc[i]
    ax.text(v, i, f' {v:.1f}% (Total: {total_abs:,.0f})'.replace('.', ','), va='center', fontsize=9, fontweight='bold')

plt.title('Porcentagem de Veículos com Tipo "ND" por Capital')
plt.xlabel('Porcentagem do Total de Veículos (%)')
plt.ylabel('Capital')
plt.legend(title='Tipo de Veículo', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Cidades a excluir
cidades_excluir = [
    #'RIO DE JANEIRO', 'TERESINA', 'VITORIA', 'RECIFE', 'GOIANIA'
    ]

# Filtrar o dataframe
df_filtered_veic = df_tipo_veiculo_capitais[~df_tipo_veiculo_capitais['municipio'].isin(cidades_excluir)]

# 1. Calcular o total de veículos envolvidos por região (filtrado)
total_veiculos_regiao_f = df_filtered_veic.groupby('regiao')['qtde_veiculos'].sum()

# 2. Calcular a quantidade dos tipos 'ND' (filtrado)
tipos_interesse = ['ND']
df_nd_f = df_filtered_veic[df_filtered_veic['tipo_veiculo'].isin(tipos_interesse)]
counts_nd_f = df_nd_f.groupby(['regiao', 'tipo_veiculo'])['qtde_veiculos'].sum().unstack(fill_value=0)

# Garantir que a coluna 'ND' exista
if 'ND' not in counts_nd_f.columns:
    counts_nd_f['ND'] = 0

# 3. Juntar os dados e calcular as porcentagens
df_calc_f = counts_nd_f.merge(total_veiculos_regiao_f.rename('Total_Geral'), left_index=True, right_index=True)

df_calc_f['% ND'] = (df_calc_f['ND'] / df_calc_f['Total_Geral']) * 100

# 4. Ordenar e preparar para plotagem
df_sorted_f = df_calc_f.sort_values('% ND', ascending=True)

# Selecionar apenas as colunas de porcentagem para o gráfico
df_plot_f = df_sorted_f[['% ND']]

# Exibir os dados tabulados (Totais e Percentuais)
print("Tabela de Dados por Região (Excluindo cidades filtradas):")
display(df_sorted_f[['Total_Geral', 'ND', '% ND']])

# 5. Plotar Gráfico
ax = df_plot_f.plot(kind='barh', stacked=True, figsize=(10, 6), colormap='Set2')

# Adicionar rótulos de porcentagem e total na ponta da barra
for i, v in enumerate(df_sorted_f['% ND']):
    total_abs = df_sorted_f['Total_Geral'].iloc[i]
    ax.text(v, i, f' {v:.1f}% (Total: {total_abs:,.0f})'.replace('.', ','), va='center', fontsize=10, fontweight='bold')

plt.title('Porcentagem de Veículos com Tipo "ND" por Região (Filtrado)')
plt.xlabel('Porcentagem do Total de Veículos (%)')
plt.ylabel('Região')
plt.legend(title='Tipo de Veículo', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4.3. Analisando o dataframe df_vitimas_capitais

In [ ]:
# Analise df_vitimas_capitais

df_vitimas_capitais = pd.read_csv("df_vitimas_capitais.csv", sep=";")
display(df_vitimas_capitais.head())

In [ ]:
df_vitimas_capitais = df_vitimas_capitais.merge(df_acidentes_capitais_raw[["municipio", "regiao", "num_acidente"]], how="left", on="num_acidente")

In [ ]:
df_vitimas_capitais.columns

In [ ]:
display(df_vitimas_capitais["susp_alcool"].unique())
display(df_vitimas_capitais["genero"].unique())
display(df_vitimas_capitais["faixa_idade"].unique())
display(df_vitimas_capitais["equip_seguranca"].unique())
display(df_vitimas_capitais["ind_motorista"].unique())
display(df_vitimas_capitais["tp_envolvido"].unique())
display(df_vitimas_capitais["gravidade_lesao"].unique())

In [ ]:
# Colunas qualitativas de interesse
cols_qualitativas = ['susp_alcool', 'genero', 'faixa_idade', 'equip_seguranca', 'ind_motorista', 'tp_envolvido', 'gravidade_lesao']

# Configurar o grid de subplots
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(18, 15))
axes = axes.flatten()

# Tipos considerados como "dados faltantes"
tipos_missing = ['NAO INFORMADO', 'DESCONHECIDO']

for i, col in enumerate(cols_qualitativas):
    ax = axes[i]

    # Contar ocorrências
    total = len(df_vitimas_capitais)
    qtd_missing = df_vitimas_capitais[col].isin(tipos_missing).sum()
    qtd_validos = total - qtd_missing

    # Calcular porcentagens
    perc_validos = (qtd_validos / total) * 100
    perc_missing = (qtd_missing / total) * 100

    # Preparar dados para o gráfico
    valores = [perc_validos, perc_missing]
    labels = ['Informado', 'N/I ou Desconhecido']
    colors = ['#66b3ff', '#ff9999'] # Azul para bom, Vermelho para ruim

    # Plotar gráfico de barras
    bars = ax.bar(labels, valores, color=colors)

    # Adicionar rótulos de dados
    ax.bar_label(bars, fmt='%.1f%%', padding=3)

    ax.set_title(f'Qualidade de Dados: {col}')
    ax.set_ylim(0, 110) # Fixar limite em 110% para caber o rótulo
    ax.set_ylabel('Porcentagem (%)')

# Remover subplots vazios se houver
for j in range(len(cols_qualitativas), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# Definir colunas e valores considerados nulos
cols_qualitativas = ['susp_alcool', 'genero', 'faixa_idade', 'equip_seguranca', 'ind_motorista', 'tp_envolvido', 'gravidade_lesao']
tipos_missing = ['NAO INFORMADO', 'DESCONHECIDO']

# Calcular a porcentagem de 'missing' para cada coluna agrupado por município e UF
# Utilizamos 'uf_acidente' pois é a coluna de estado garantida no dataframe de vítimas
quality_matrix = df_vitimas_capitais.groupby(['municipio', 'uf_acidente'])[cols_qualitativas].apply(lambda x: x.isin(tipos_missing).mean() * 100)

# Exibir a matriz numérica (opcional, mas útil para ver detalhes)
# display(quality_matrix)

# Plotar o Heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(quality_matrix, annot=True, cmap='viridis_r', fmt='.1f', linewidths=.5)

plt.title('Porcentagem de Dados "Não Informado" ou "Desconhecido" por Capital e Variável')
plt.ylabel('Capital - UF')
plt.xlabel('Variável Qualitativa')
plt.tight_layout()
plt.show()

In [ ]:
# Cidades a serem excluídas da análise
cidades_excluir = [
    #'RIO DE JANEIRO', 'TERESINA', 'VITORIA', 'RECIFE', 'GOIANIA'
    ]

# Filtrar o dataframe
df_vitimas_filtered = df_vitimas_capitais[~df_vitimas_capitais['municipio'].isin(cidades_excluir)]

# Calcular a porcentagem de 'missing' por região com os dados filtrados
quality_matrix_regiao_filtered = df_vitimas_filtered.groupby('regiao')[cols_qualitativas].apply(lambda x: x.isin(tipos_missing).mean() * 100)

# Plotar o Heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(quality_matrix_regiao_filtered, annot=True, cmap='viridis_r', fmt='.1f', linewidths=.5)

plt.title('Porcentagem de Dados "Não Informado" ou "Desconhecido" por Região (Excluindo RJ, TE, VIX, RE, GO)')
plt.ylabel('Região')
plt.xlabel('Variável Qualitativa')
plt.tight_layout()
plt.show()

## 4.4. Criando o dataset final

In [ ]:
# Ajustar o dataset final com as features necessárias.

display(df_acidentes_capitais_raw.head())
display(df_vitimas_capitais.head())
display(df_tipo_veiculo_capitais.head())

In [ ]:
display(df_tipo_veiculo_capitais[df_tipo_veiculo_capitais["qtde_veiculos"] > 5].head())
display(df_tipo_veiculo_capitais[df_tipo_veiculo_capitais["num_acidente"]==301282])

In [ ]:
# Pivotando o dataframe de veículos
df_veiculos_pivot = df_tipo_veiculo_capitais.pivot_table(
    index='num_acidente',
    columns='tipo_veiculo',
    values='qtde_veiculos',
    aggfunc='sum',
    fill_value=0,
).reset_index()

# Removendo o nome da hierarquia das colunas para deixar mais limpo
df_veiculos_pivot.columns.name = None

display(df_veiculos_pivot[df_veiculos_pivot["num_acidente"]==301282])

In [ ]:
display(df_tipo_veiculo_capitais.shape)
display(df_veiculos_pivot.shape)

In [ ]:
# Separando colunas quantitativas e qualitativas
df_quantitativas = df_acidentes_capitais_raw.select_dtypes(include=['number'])
df_qualitativas = df_acidentes_capitais_raw.select_dtypes(exclude=['number'])

list_quantitativas = df_quantitativas.columns.tolist()

list_qualitativas = df_qualitativas.columns.tolist()

display(list_quantitativas)
display(list_qualitativas)

In [ ]:
# Definindo valores considerados como dados faltantes para variáveis qualitativas
valores_missing_qual = ['NAO INFORMADO', 'DESCONHECIDO', 'IGNORADO', 'NA', 'DESCONHECIDAS', 'DESCONHECIDA', 'ND']

# Calculando a porcentagem de missing values para as colunas qualitativas
# Verifica se está na lista de missing ou se é nulo/NaN
missing_pct_qual = df_acidentes_capitais_raw[list_qualitativas].apply(
    lambda x: (x.isin(valores_missing_qual) | x.isnull()).mean() * 100
)

# Criando um DataFrame para melhor visualização
df_missing_qual_summary = pd.DataFrame({
    'Coluna': missing_pct_qual.index,
    '% Missing': missing_pct_qual.values
}).sort_values(by='% Missing', ascending=False)

# Exibindo o resultado
print("Porcentagem de Dados Faltantes em Colunas Qualitativas:")
display(df_missing_qual_summary)

# Visualização gráfica das Top 15 colunas com mais dados faltantes
plt.figure(figsize=(10, 8))
sns.barplot(x='% Missing', y='Coluna', data=df_missing_qual_summary.head(15), palette='magma')
plt.title('Top 15 Colunas Qualitativas com Mais Dados Faltantes')
plt.xlabel('Porcentagem de Missing (%)')
plt.ylabel('Coluna')
plt.xlim(0, 100)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Identificar colunas com mais de 25% de missing values
cols_to_drop = df_missing_qual_summary[df_missing_qual_summary['% Missing'] > 25]['Coluna'].tolist()

print(f"Colunas a serem removidas ({len(cols_to_drop)}):")
display(cols_to_drop)

# Remover as colunas do dataframe principal
df_acidentes_capitais_raw.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# Atualizar a lista de colunas qualitativas
df_qualitativas = df_acidentes_capitais_raw.select_dtypes(exclude=['number'])
list_qualitativas = df_qualitativas.columns.tolist()

print("\nColunas qualitativas restantes:")
display(list_qualitativas)

In [ ]:
# Calcular a porcentagem de missing values para as colunas qualitativas restantes por capital
# Utiliza a lista 'list_qualitativas' atualizada e 'valores_missing_qual' já definidos
missing_by_capital_qual = df_acidentes_capitais_raw.groupby(['municipio', 'uf'])[list_qualitativas].apply(
    lambda x: (x.isin(valores_missing_qual) | x.isnull()).mean() * 100
)

# Exibir a tabela
print("Porcentagem de Dados Faltantes por Capital (Variáveis Qualitativas Restantes):")
display(missing_by_capital_qual)

# Visualizar com um Heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(missing_by_capital_qual, annot=True, cmap='viridis_r', fmt='.1f', vmin=0, vmax=100)
plt.title('Porcentagem de Dados "Não Informado/Nulo" por Capital (Qualitativas Restantes)')
plt.ylabel('Capital - UF')
plt.xlabel('Variável Qualitativa')
plt.tight_layout()
plt.show()

In [ ]:
# Colunas adicionais a serem removidas conforme solicitado
cols_manual_drop = ['chv_localidade', 'uf_acidente', 'end_acidente', 'bairro_acidente']

# Remover as colunas do dataframe principal
df_acidentes_capitais_raw.drop(columns=cols_manual_drop, inplace=True, errors='ignore')

# Atualizar a lista de colunas qualitativas
df_qualitativas = df_acidentes_capitais_raw.select_dtypes(exclude=['number'])
list_qualitativas = df_qualitativas.columns.tolist()

print(f"Colunas removidas manualmente: {cols_manual_drop}")
print("\nColunas qualitativas restantes:")
display(list_qualitativas)

In [ ]:
list_quantitativas

In [ ]:
# Colunas quantitativas a serem removidas conforme solicitado
cols_quant_drop = ['codigo_ibge', 'num_end_acidente', 'cep_acidente', 'km_via_acidente', 'latitude_acidente', 'longitude_acidente']

# Remover as colunas do dataframe principal
df_acidentes_capitais_raw.drop(columns=cols_quant_drop, inplace=True, errors='ignore')

# Atualizar a lista de colunas quantitativas
df_quantitativas = df_acidentes_capitais_raw.select_dtypes(include=['number'])
list_quantitativas = df_quantitativas.columns.tolist()

print(f"Colunas quantitativas removidas: {cols_quant_drop}")
print("\nColunas quantitativas restantes:")
display(list_quantitativas)

In [ ]:
# Combinando as listas de colunas a manter
cols_to_keep = list_quantitativas + list_qualitativas

# Filtrando o dataframe principal para garantir que tenha apenas essas colunas
df_acidentes_capitais_raw = df_acidentes_capitais_raw[cols_to_keep]

# Exibindo o resultado
print(f"Total de colunas mantidas: {len(cols_to_keep)}")
display(df_acidentes_capitais_raw.head())
display(df_acidentes_capitais_raw.info())

In [ ]:
# Analisando o formato máximo para inferir se é HHMM ou HHMMSS
print(f"Valor máximo em hora_acidente: {df_acidentes_capitais_raw['hora_acidente'].max()}")

# Convertendo para string e preenchendo com zeros à esquerda para garantir 6 dígitos (HHMMSS)
# Assume-se que o formato original numérico seja HHMMSS (ex: 143000 = 14:30:00)
df_acidentes_capitais_raw['hora_str'] = df_acidentes_capitais_raw['hora_acidente'].astype(str).str.zfill(6)

# Convertendo para objeto datetime e extraindo o tempo
df_acidentes_capitais_raw['horario'] = pd.to_datetime(df_acidentes_capitais_raw['hora_str'], format='%H%M%S', errors='coerce').dt.time

# Verificando a conversão
display(df_acidentes_capitais_raw[['hora_acidente', 'hora_str', 'horario']].head())

# Substituindo a coluna original ou mantendo a nova (conforme preferência, aqui mantendo 'horario' e removendo auxiliares)
df_acidentes_capitais_raw.drop(columns=['hora_str'], inplace=True)

# Verificar se houve falhas na conversão (NaT)
nulos_horario = df_acidentes_capitais_raw['horario'].isnull().sum()
print(f"Falhas na conversão de horário: {nulos_horario}")

## 4.5. Ajustando o df_final

In [ ]:
df_final_raw = df_acidentes_capitais_raw.copy()

In [ ]:
df_final_raw.shape

In [ ]:
# Restaurando o dataframe base para garantir o estado inicial antes do filtro
df_final_raw = df_acidentes_capitais_raw.copy()

# Cidades a serem removidas por baixa qualidade de dados
# Salvador: Problemas consistentes com horário (999999)
# Rio de Janeiro: Alto índice de 'NAO INFORMADO' e 'DESCONHECIDO' em tipo de veículo
cidades_remover = ['SALVADOR', 'RIO DE JANEIRO']

# Contar linhas antes
rows_before = len(df_final_raw)

# Filtrar o dataframe excluindo as cidades
df_final_raw_ajustado = df_final_raw[~df_final_raw['municipio'].isin(cidades_remover)]

# Contar linhas depois
rows_after = len(df_final_raw_ajustado)

print(f"Cidades removidas: {cidades_remover}")
print(f"Linhas removidas: {rows_before - rows_after}")
print(f"Novo shape do dataframe: {df_final_raw.shape}")

In [ ]:
df_final = df_final_raw.merge(df_veiculos_pivot, how="left", on="num_acidente")

In [ ]:
df_final_ajustado = df_final_raw_ajustado.merge(df_veiculos_pivot, how="left", on="num_acidente")

In [ ]:
print(f"Tamanho do dataframe final: {len(df_final)} \n")
print(f"Tamanho do dataframe final ajustado: {len(df_final_ajustado)}")

## 4.6 Normalizando os datasets

In [ ]:
# Lista de valores a serem considerados como 'missing' e substituídos por 'ND'
valores_para_nd = ['NAO INFORMADO', 'DESCONHECIDO', 'IGNORADO', 'NA', 'DESCONHECIDAS', 'DESCONHECIDA', '']

# Processar df_final
print("Processando df_final...")
for col in df_final.select_dtypes(include='object').columns:
    df_final[col] = df_final[col].replace(valores_para_nd, 'ND')
    df_final[col] = df_final[col].fillna('ND') # Tratar NaNs também como ND

# Processar df_final_ajustado
print("Processando df_final_ajustado...")
for col in df_final_ajustado.select_dtypes(include='object').columns:
    df_final_ajustado[col] = df_final_ajustado[col].replace(valores_para_nd, 'ND')
    df_final_ajustado[col] = df_final_ajustado[col].fillna('ND') # Tratar NaNs também como ND

print("Normalização concluída para df_final e df_final_ajustado.")

In [ ]:
df_final.columns

In [ ]:
cols_to_analyze = ['qtde_acidente', 'qtde_acid_com_obitos', 'qtde_envolvidos', 'qtde_feridosilesos', 'qtde_obitos']

print("Estatísticas Descritivas para as colunas selecionadas:")
display(df_final[cols_to_analyze].describe())

In [ ]:
# Identificar as colunas de tipo de veículo que foram pivotadas
# A lista de colunas do df_veiculos_pivot (exceto 'num_acidente') representa os tipos de veículos.
vehicle_type_cols = [col for col in df_veiculos_pivot.columns if col != 'num_acidente']

# Criar um mapeamento para renomear as colunas
rename_mapping = {col: 'tp_veiculo_' + col for col in vehicle_type_cols}

# Aplicar o renomeamento ao df_final
df_final.rename(columns=rename_mapping, inplace=True)

# Aplicar o renomeamento ao df_final_ajustado
df_final_ajustado.rename(columns=rename_mapping, inplace=True)

print("Colunas de tipo de veículo renomeadas com sucesso.")
print("Novas colunas de df_final:")
display(df_final.columns)

In [ ]:
# Replace spaces in column names with underscores for df_final
df_final.columns = df_final.columns.str.replace(' ', '_')

# Replace spaces in column names with underscores for df_final_ajustado
df_final_ajustado.columns = df_final_ajustado.columns.str.replace(' ', '_')

print("Espaços nos nomes das colunas de df_final foram substituídos por underscores.")
display(df_final.columns)

print("\nEspaços nos nomes das colunas de df_final_ajustado foram substituídos por underscores.")
display(df_final_ajustado.columns)

In [ ]:
# Renomear a coluna 'tp_veiculo_MOTORCASA(MOTORHOME)' para 'tp_veiculo_MOTORHOME' no df_final
if 'tp_veiculo_MOTORCASA(MOTORHOME)' in df_final.columns:
    df_final.rename(columns={'tp_veiculo_MOTORCASA(MOTORHOME)': 'tp_veiculo_MOTORHOME'}, inplace=True)
    print("Coluna 'tp_veiculo_MOTORCASA(MOTORHOME)' renomeada para 'tp_veiculo_MOTORHOME' em df_final.")
else:
    print("Coluna 'tp_veiculo_MOTORCASA(MOTORHOME)' não encontrada em df_final.")

# Renomear a coluna 'tp_veiculo_MOTORCASA(MOTORHOME)' para 'tp_veiculo_MOTORHOME' no df_final_ajustado
if 'tp_veiculo_MOTORCASA(MOTORHOME)' in df_final_ajustado.columns:
    df_final_ajustado.rename(columns={'tp_veiculo_MOTORCASA(MOTORHOME)': 'tp_veiculo_MOTORHOME'}, inplace=True)
    print("Coluna 'tp_veiculo_MOTORCASA(MOTORHOME)' renomeada para 'tp_veiculo_MOTORHOME' em df_final_ajustado.")
else:
    print("Coluna 'tp_veiculo_MOTORCASA(MOTORHOME)' não encontrada em df_final_ajustado.")

print("\nNovas colunas de df_final (primeiras 50):")
display(df_final.columns[:50])
print("\nNovas colunas de df_final_ajustado (primeiras 50):")
display(df_final_ajustado.columns[:50])

In [ ]:
# Converter todas as colunas de df_final para letras minúsculas
df_final.columns = df_final.columns.str.lower()

# Converter todas as colunas de df_final_ajustado para letras minúsculas
df_final_ajustado.columns = df_final_ajustado.columns.str.lower()

print("Todas as colunas de df_final foram convertidas para letras minúsculas.")
display(df_final.columns)

print("\nTodas as colunas de df_final_ajustado foram convertidas para letras minúsculas.")
display(df_final_ajustado.columns)

## 4.7. Retornando para a base de vítimas

In [ ]:
df_vitimas_capitais.head()

In [ ]:
display(df_vitimas_capitais["num_acidente"].count())
display(df_vitimas_capitais["num_acidente"].unique().size)

In [ ]:
# Lista de valores a serem substituídos por 'ND'
valores_para_nd = ['NAO INFORMADO', 'DESCONHECIDO', 'IGNORADO', 'NA', 'DESCONHECIDAS', 'DESCONHECIDA', '']

# Processar df_vitimas_capitais
print("Processando df_vitimas_capitais...")
for col in df_vitimas_capitais.select_dtypes(include='object').columns:
    df_vitimas_capitais[col] = df_vitimas_capitais[col].replace(valores_para_nd, 'ND')
    df_vitimas_capitais[col] = df_vitimas_capitais[col].fillna('ND') # Tratar NaNs também como ND

print("Normalização concluída para df_vitimas_capitais.")

# Verificar o resultado em algumas colunas
print("\nValores únicos após normalização (exemplo: gravidade_lesao):")
display(df_vitimas_capitais['gravidade_lesao'].unique())

In [ ]:
# Definindo as colunas qualitativas de interesse para análise
cols_qualitativas_vitimas = ['faixa_idade', 'genero', 'tp_envolvido', 'gravidade_lesao', 'equip_seguranca', 'ind_motorista', 'susp_alcool']

# Loop para analisar cada coluna
for col in cols_qualitativas_vitimas:
    if col in df_vitimas_capitais.columns:
        print(f"--- Distribuição da coluna: {col} ---")

        # Calculando a distribuição percentual
        distribuicao = df_vitimas_capitais[col].value_counts(normalize=True) * 100
        display(distribuicao.head(10)) # Exibindo as 10 categorias mais frequentes

        # Visualização Gráfica
        plt.figure(figsize=(10, 6))
        sns.countplot(data=df_vitimas_capitais, y=col, order=df_vitimas_capitais[col].value_counts().index[:15], palette='viridis')
        plt.title(f'Distribuição de Valores - {col}')
        plt.xlabel('Contagem')
        plt.ylabel(col)
        plt.tight_layout()
        plt.show()
        print("\n")
    else:
        print(f"A coluna {col} não foi encontrada no dataframe.")

In [ ]:
# Removendo colunas do df_vitimas_capitais
cols_to_drop_vitimas = ['gravidade_lesao', 'equip_seguranca', 'faixa_idade', 'Unnamed: 0']

df_vitimas_capitais.drop(columns=cols_to_drop_vitimas, inplace=True, errors='ignore')

print(f"Colunas removidas: {cols_to_drop_vitimas}")
print("Colunas restantes:")
display(df_vitimas_capitais.columns)

In [ ]:
# Colunas a serem mantidas no df_vitimas_capitais
cols_to_keep_vitimas = ['num_acidente', 'genero', 'susp_alcool', 'ind_motorista']

# Filtrar o dataframe para manter apenas essas colunas
df_vitimas_capitais = df_vitimas_capitais[cols_to_keep_vitimas]

print("Colunas mantidas no df_vitimas_capitais:")
display(df_vitimas_capitais.columns)
display(df_vitimas_capitais.head())

In [ ]:
df_vitimas_capitais['genero'].unique()

In [ ]:
# Colunas para pivotar
cols_to_pivot = ['genero', 'ind_motorista', 'susp_alcool']

# Verificar se as colunas existem no dataframe
cols_present = [col for col in cols_to_pivot if col in df_vitimas_capitais.columns]

print(f"Pivotando as colunas: {cols_present}")

# Criar dummies (one-hot encoding) e agrupar por número do acidente
# Isso conta quantas vítimas de cada característica existem por acidente
df_vitimas_pivot = pd.get_dummies(df_vitimas_capitais[cols_present], prefix=cols_present)
df_vitimas_pivot['num_acidente'] = df_vitimas_capitais['num_acidente']
df_vitimas_pivot = df_vitimas_pivot.groupby('num_acidente').sum().reset_index()

# Converter colunas para minúsculas
df_vitimas_pivot.columns = df_vitimas_pivot.columns.str.lower()

# Exibir o resultado
display(df_vitimas_pivot.head())
print(f"Dimensões do novo dataframe pivotado: {df_vitimas_pivot.shape}")

In [ ]:
df_final = df_final.merge(df_vitimas_pivot, how="left", on="num_acidente")
df_final_ajustado = df_final_ajustado.merge(df_vitimas_pivot, how="left", on="num_acidente")

In [ ]:
display(df_final.shape)
display(df_final_ajustado.shape)

In [ ]:
df_final.to_csv("df_final.csv", sep=";")
df_final_ajustado.to_csv("df_final_ajustado.csv", sep=";")